In [ ]:
#| default_exp ssm_analysis

# SSM Analysis

> Self-Similarity Matrix analysis of midi-rae encoder representations.
>
> For each song, collects all 128x128 crop embeddings from preencoded chunks,
> computes pairwise cosine similarity at each encoder level, and visualizes as
> heatmaps. Also computes pixel-space SSM for comparison.
>
> Key hypothesis: embedding-space SSM should show cleaner block structure
> (verse/chorus repetition) than pixel-space SSM.

In [ ]:
#| export
import glob
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import pearsonr

In [ ]:
#| export
def load_song_embeddings(encoded_dir, song_idx, level=5, split='train'):
    """Load all crop embeddings for a song from preencoded chunks.

    Each chunk item has emb1: list of 6 level tensors [B, N_patches, D].
    Returns mean-pooled embeddings [N_crops, D] and matching images [N_crops, 1, H, W].
    """
    encoded_dir = os.path.expanduser(encoded_dir)
    files = sorted(glob.glob(os.path.join(encoded_dir, f'{split}_chunk*.pt')))
    assert files, f"No {split}_chunk*.pt found in {encoded_dir}"

    embs, imgs = [], []
    for f in files:
        data = torch.load(f, map_location='cpu', weights_only=False)
        for item in data:
            file_idxs = item['file_idx']
            emb = item['emb1'][level].float()   # [B, N_patches, D]
            img = item['img1'].float()           # [B, 1, H, W]
            mask = (file_idxs == song_idx)
            if mask.any():
                embs.append(emb[mask].mean(dim=1))  # mean-pool patches
                imgs.append(img[mask])

    if not embs:
        return None, None
    return torch.cat(embs, dim=0), torch.cat(imgs, dim=0)

In [ ]:
#| export
def compute_ssm(vecs):
    "Pairwise cosine similarity matrix. vecs: [N, D] -> [N, N] numpy array."
    normed = F.normalize(vecs, dim=-1)
    return (normed @ normed.T).numpy()


def compute_pixel_ssm(imgs):
    "Pixel-space SSM: flatten each image, cosine similarity. imgs: [N, 1, H, W]."
    flat = imgs.flatten(1).float()
    return compute_ssm(flat)

In [ ]:
#| export
def plot_ssm_alllevels(encoded_dir, song_idx, split='train', out_dir=None, show=True):
    """Plot SSM for all 6 encoder levels + pixel space for a single song.

    Returns (ssms dict, pearson_rs dict) keyed by level (0-5) and 'pixel'.
    """
    _, imgs = load_song_embeddings(encoded_dir, song_idx, level=5, split=split)
    if imgs is None:
        print(f"Song {song_idx} not found in {split} split.")
        return None, None
    pixel_ssm = compute_pixel_ssm(imgs)

    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    fig.suptitle(f"SSM -- Song {song_idx:03d} ({len(imgs)} crops)", fontsize=14)

    # Pixel SSM in last panel
    ax = axes[1, 3]
    ax.imshow(pixel_ssm, cmap='viridis', vmin=-1, vmax=1, aspect='auto')
    ax.set_title("Pixel space")
    ax.set_xlabel('Crop'); ax.set_ylabel('Crop')

    ssms = {'pixel': pixel_ssm}
    pearson_rs = {}

    for lvl in range(6):
        row, col = divmod(lvl, 4)
        ax = axes[row, col]
        embs, _ = load_song_embeddings(encoded_dir, song_idx, level=lvl, split=split)
        if embs is None:
            ax.set_title(f"L{lvl} -- no data")
            continue
        ssm = compute_ssm(embs)
        ssms[lvl] = ssm
        r, _ = pearsonr(ssm.flatten(), pixel_ssm.flatten())
        pearson_rs[lvl] = r
        ax.imshow(ssm, cmap='viridis', vmin=-1, vmax=1, aspect='auto')
        ax.set_title(f"L{lvl} ({embs.shape[1]}d)  r={r:.3f}")
        ax.set_xlabel('Crop'); ax.set_ylabel('Crop')

    plt.tight_layout()
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, f"ssm_song{song_idx:03d}.png")
        plt.savefig(path, dpi=150)
        print(f"Saved: {path}")
    if show:
        plt.show()
    else:
        plt.close()

    return ssms, pearson_rs

In [ ]:
#| export
def ssm_survey(encoded_dir, song_indices, split='train', out_dir='./results/ssm'):
    """Run SSM analysis over multiple songs, collect Pearson r values per level.

    Returns {level: [r_song1, r_song2, ...]} for levels 0-5.
    """
    all_rs = {lvl: [] for lvl in range(6)}
    for song_idx in song_indices:
        print(f"Song {song_idx:03d}...", end=' ', flush=True)
        _, rs = plot_ssm_alllevels(encoded_dir, song_idx, split=split,
                                    out_dir=out_dir, show=False)
        if rs is None:
            print("skipped")
            continue
        for lvl, r in rs.items():
            all_rs[lvl].append(r)
        print(" ".join(f"L{l}={rs.get(l, float('nan')):.3f}" for l in range(6)))

    print("\n--- Mean Pearson r (embedding SSM vs pixel SSM) ---")
    for lvl in range(6):
        vals = all_rs[lvl]
        if vals:
            print(f"  L{lvl}: {np.mean(vals):.3f} +/- {np.std(vals):.3f} (n={len(vals)})")
    return all_rs

## Run analysis

In [ ]:
ENCODED_DIR = '~/datasets/POP909_encoded_exp26'
OUT_DIR = './results/ssm'

# Single song -- full all-levels plot
ssms, rs = plot_ssm_alllevels(ENCODED_DIR, song_idx=1, out_dir=OUT_DIR, show=True)
print("Pearson r per level:", rs)

In [ ]:
# Survey across 20 random songs
import random
random.seed(42)
sample_songs = random.sample(range(909), 20)
all_rs = ssm_survey(ENCODED_DIR, sample_songs, out_dir=OUT_DIR)

In [ ]:
#| export
#| eval: false
import argparse

def ssm_main():
    "CLI entry point: run SSM survey (or single-song plot) from preencoded chunks."
    parser = argparse.ArgumentParser(description='SSM analysis of midi-rae encoder representations')
    parser.add_argument('--encoded_dir', default='~/datasets/POP909_encoded_exp26')
    parser.add_argument('--out_dir',     default='./results/ssm')
    parser.add_argument('--split',       default='train')
    parser.add_argument('--song_idx',    type=int, default=None, help='single song to plot (default: run survey)')
    parser.add_argument('--n_songs',     type=int, default=20,   help='number of songs for survey')
    parser.add_argument('--seed',        type=int, default=42)
    args = parser.parse_args()

    import random, numpy as np
    random.seed(args.seed)

    if args.song_idx is not None:
        ssms, rs = plot_ssm_alllevels(args.encoded_dir, args.song_idx,
                                      split=args.split, out_dir=args.out_dir, show=False)
        print('Pearson r per level:', rs)
    else:
        sample_songs = random.sample(range(909), args.n_songs)
        ssm_survey(args.encoded_dir, sample_songs, split=args.split, out_dir=args.out_dir)

if __name__ == '__main__' and 'ipykernel' not in __import__('sys').modules:
    ssm_main()